# ⚡ Imagen → 3D CON TEXTURA — Stable-Fast-3D (versión limpia)

Genera un modelo 3D **ya con color** (con los colores de tu imagen) en la T4 gratis de Colab.
Esta versión ya tiene **todos los parches metidos adentro** — no hay que andar pegando código suelto.

## Requisito único (una sola vez, gratis)
1. Cuenta gratis en https://huggingface.co
2. Entrá a **https://huggingface.co/stabilityai/stable-fast-3d** → botón **"Agree and access repository"**
3. Token en **https://huggingface.co/settings/tokens** → "New token" → tipo **Read** → copialo (`hf_...`)

## Orden — seguí los números EN ORDEN, sin saltear ni repetir
1. GPU: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU**.
2. **Celda 1** (instalar, ~5 min).
3. Cuando termine: **`Entorno de ejecución` → `Reiniciar sesión`** (una sola vez, obligatorio).
4. **Celda 2** (verifica todo + pegás tu token).
5. **Celda 3** (subís tu imagen).
6. **Celda 4** (genera el modelo).
7. **Celda 5** (descarga el `.glb`).

⚠️ Si en algún momento **Colab te desconecta o decís "Reiniciar sesión" de nuevo por tu cuenta**, se borra todo `/content`
→ volvé derecho a la **Celda 1** y arrancá de nuevo (rápido, ya queda cacheado).

## Celda 1 — Instalar todo (con los parches de numpy/scipy/rembg ya incluidos)

In [ ]:
!nvidia-smi -L

import os
os.chdir('/content')
if not os.path.isdir('/content/stable-fast-3d'):
    !git clone https://github.com/Stability-AI/stable-fast-3d.git
os.chdir('/content/stable-fast-3d')

!pip install -q "setuptools==69.5.1" wheel
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip install -q ./texture_baker ./uv_unwrapper 2>&1 | tail -2

# rembg no viene en requirements.txt pero run.py lo necesita para quitar el fondo.
# Se instala junto con numpy/scipy en un rango que combina con TODO (rembg pide numpy>=2.3,
# numba/cupy piden numpy<2.5) para no tener que parchear numpy dos veces.
!pip install -q rembg onnxruntime "numpy>=2.3.0,<2.5" "scipy>=1.16.3" 2>&1 | tail -5

print('\n✅ Celda 1 terminada.')
print('⚠️  AHORA: Entorno de ejecución -> Reiniciar sesión (una sola vez).')
print('   Después seguí con la Celda 2. NO vuelvas a correr esta Celda 1.')

## Celda 2 — Verificar todo + iniciar sesión en Hugging Face
Corré esta celda recién DESPUÉS de reiniciar la sesión. Te va a pedir el token (`hf_...`).

In [ ]:
import torch, numpy, scipy, rembg
print('torch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print('numpy:', numpy.__version__, '| scipy:', scipy.__version__, '| rembg OK')

from getpass import getpass
from huggingface_hub import login
tok = getpass('Pegá tu token hf_... y Enter: ')
login(token=tok.strip())
print('\n✅ Todo listo. Seguí con la Celda 3.')

## Celda 3 — Subir tu imagen
Cualquier imagen del personaje **de frente** (SF3D le quita el fondo solo). Si el botón no anda en el celular,
subila por el panel **Archivos** 📁 (carpeta de la izquierda) y corré esta celda igual — la detecta sola.

In [ ]:
import os, glob
from PIL import Image

IMG = None
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMG = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget no anduvo (', e, ') -> uso el panel Archivos.')

if not IMG or not os.path.exists(IMG):
    cand = []
    for ext in ('png','jpg','jpeg','webp'):
        cand += glob.glob('/content/*.'+ext)
    cand.sort(key=os.path.getmtime)
    IMG = cand[-1] if cand else None

assert IMG and os.path.exists(IMG), 'No encontré imagen. Subila por el botón o por el panel Archivos 📁 y volvé a correr esta celda.'
im = Image.open(IMG)
print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)
print('✅ Lista. Seguí con la Celda 4.')

## Celda 4 — Generar el modelo 3D texturizado
La primera vez baja el modelo (~2 GB). Sale `mesh.glb` **con textura**.

In [ ]:
import os
assert os.path.isdir('/content/stable-fast-3d'), '⚠️ Falta la Celda 1 (la sesión se reinició sola) — volvé a correrla desde el principio.'
os.chdir('/content/stable-fast-3d')
os.makedirs('output', exist_ok=True)

!python run.py "{IMG}" --output-dir output/ --texture-resolution 1024

OUT = None
for root, dirs, fs in os.walk('output'):
    for f in fs:
        if f.endswith('.glb'):
            OUT = os.path.join(root, f)
print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024/1024, 2)) + ' MB (texturizado)')
      if OUT else '❌ no se generó — copiame el error rojo de arriba')

## Celda 5 — Descargar el `.glb` texturizado
Probalo en https://gltf-viewer.donmccurdy.com — se ve con colores, sin pasos extra.

In [ ]:
from google.colab import files
import os
for root, dirs, fs in os.walk('/content/stable-fast-3d/output'):
    for f in fs:
        if f.endswith('.glb'):
            files.download(os.path.join(root, f))

---
### Errores conocidos (por si aparecen)
- **`401` / `gated repo`** → te faltó aceptar la licencia en https://huggingface.co/stabilityai/stable-fast-3d, o el token no es válido → volvé a correr la Celda 2.
- **`CUDA out of memory`** → en la Celda 4 cambiá `--texture-resolution 1024` por `512`.
- **`FileNotFoundError: /content/stable-fast-3d`** → la sesión se reinició sola (por inactividad o desconexión) → volvé a la **Celda 1**.
- **La espalda sale inventada** → normal, SF3D solo ve la vista de frente. Para la espalda fiel a tu dibujo: Space **MV-Adapter Img2Texture** (https://huggingface.co/spaces/VAST-AI/MV-Adapter-Img2Texture) con tu malla + tu imagen.
- Cualquier error rojo nuevo, copiámelo. ⚡